# Week 4, Day 3: Port a Royal Rumble simulator to C++

**Lesson:** [week4/day3.ipynb](../../../week4/day3.ipynb), *Code generator: Python to fast C++*. **About 40 minutes.**

**What you practice here**

- Writing a system prompt and a user prompt that make a model convert code: C++ only, identical output, and the context it needs, which is your machine's details and the compile command.
- Making the porting call yourself with `client.chat.completions.create`, asking GPT models for high reasoning effort, and taking the code out of the reply without its markdown fences.
- Reaching OpenAI, Claude, Gemini and Grok through the same `OpenAI` class, each at its own base URL, to compare frontier models on one task.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week4/day3", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson asks frontier models to port a Python program to C++ that "produces identical output in the least time",
then compiles each port and compares the speed-ups. Do the same here for a Royal Rumble simulator: 30 entrants, a
fixed-seed generator decides who goes over the top rope, and the program counts which entry numbers win. It is given
below, as the lesson gives `pi`.

You write the prompts, the call that asks for the port, and the clients for four frontier providers. The function
that calls the model, `port`, takes the client as its first parameter and makes the Chat Completions call itself,
exactly as you would on `OpenAI()`. The checks hand it a stand-in shaped like the real client: it records what you
send and answers with a canned reply, so nothing is spent. Compiling and running the C++ comes in Try it for real, and
it is optional there, as it is in the lesson.

## What each call should give

```python
compile_command = ["g++", "-std=c++17", "-O3", "-march=native", "main.cpp", "-o", "main"]
messages = messages_for(RUMBLE, retrieve_system_info(), compile_command)
# [{'role': 'system', 'content': SYSTEM_PROMPT},
#  {'role': 'user', 'content': 'Port this Python code to C++ ... with identical output ...\n'
#                              "The system information is:\n{'os': {'system': 'Linux', ...}, ...}\n"
#                              "... compiled with this command:\n['g++', '-std=c++17', ...]\n\n"
#                              '```python\n\nimport time\n...\n```'}]

clients = frontier_clients({"OPENAI_API_KEY": "sk-proj-...", "GROK_API_KEY": "xai-..."})
sorted(clients)              # ['grok', 'openai']
clients["grok"].base_url     # URL('https://api.x.ai/v1/')

port(clients["openai"], messages)
# makes one request with the model gpt-5, those messages and high reasoning effort;
# a reply of '```cpp\n#include <cstdio>\n...\n```' comes back as '#include <cstdio>\n...'

port(clients["grok"], messages, model="grok-4-fast-non-reasoning")
# the same request, sent to Grok with that model and no reasoning effort
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
from openai import OpenAI

## Given

The program you port: a Royal Rumble simulator, as the lesson gives `pi`. Run this cell. Try it for real uses
`RUMBLE`, and so does Day 4; the parts don't need it.

In [ ]:
RUMBLE = """
import time

def next_state(state):
    return (1103515245 * state + 12345) % 2**31

def rumble_winners(rumbles, entrants, seed):
    wins = [0] * (entrants + 1)
    state = seed
    for _ in range(rumbles):
        ring = []
        for number in range(1, entrants + 1):
            ring.append(number)
            state = next_state(state)
            if len(ring) > 1 and (state >> 16) % 3 != 0:  # two times in three, someone goes over the top rope
                state = next_state(state)
                ring.pop((state >> 16) % len(ring))
        while len(ring) > 1:  # after the last entrant, down to one winner
            state = next_state(state)
            ring.pop((state >> 16) % len(ring))
        wins[ring[0]] += 1
    return wins

start_time = time.time()
wins = rumble_winners(500_000, 30, 1988)
best = max(range(1, 31), key=lambda number: wins[number])
end_time = time.time()

print(f"Entry 1 won {wins[1]} times; entry 30 won {wins[30]} times")
print(f"Most wins: entry number {best}, with {wins[best]}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

### 1 · `SYSTEM_PROMPT`

The system prompt that makes the model a converter: Python in, fast C++ out. Like the lesson's `system_prompt`, it
gives the task (convert Python code into high-performance C++), the shape of the answer (C++ code only, with no
explanation other than occasional comments, so the reply can go straight into `main.cpp`), and the bar to meet
(output identical to the Python program's, in the fastest possible time). The wording is yours, in a string of 20
characters or more, but it must name `C++` and use the words "only" and "identical", which the checks look for.

<details><summary>Hint</summary>

A triple-quoted string, as the lesson's `system_prompt` is: the task, then the shape of the answer, then the bar.

```python
SYSTEM_PROMPT = """Your task is to ... (convert Python into high-performance C++)
Respond only with ... (the shape of the answer)
... identical output ... (the bar to meet)"""
```

</details>

In [ ]:
SYSTEM_PROMPT = ""

In [ ]:
check("SYSTEM_PROMPT")

### 2 · `messages_for(python, system_info, compile_command)`

Builds the messages for porting one program: your system prompt, then a user prompt with the program and your
machine's details. `python` is the program's source, `system_info` is the dict the lesson's `retrieve_system_info()`
returns, and `compile_command` is a list of strings, such as `["g++", "-O3", "main.cpp", "-o", "main"]`. Return a new
list of two messages, each a dict with a `"role"` and a `"content"`: a `"system"` message whose content is
`SYSTEM_PROMPT`, then a `"user"` message, like the lesson's `user_prompt_for`, that asks for the fastest C++ port with
identical output and holds three things:

- `system_info`, as text;
- every item of `compile_command`;
- the Python code in a fenced block: a line of three backticks followed by `python`, the code, then a line of three
  backticks.

As the lesson does, you may also say that the reply will be written to `main.cpp` and compiled with that command.

<details><summary>Hint</summary>

An f-string puts a dict or a list in as text. The lesson's user prompt, with its global names made parameters:

```python
user = (f"Port this Python code to C++ ... identical output ...\n"
        f"The system information is:\n{system_info}\n"
        f"... compiled with this command:\n{compile_command}\n\n"
        f"```python\n{python}\n```")
```

</details>

In [ ]:
def messages_for(python, system_info, compile_command):
    # Write messages_for here, as described above.
    raise NotImplementedError("messages_for is not written yet")

In [ ]:
check("messages_for")

### 3 · `port(client, messages, model="gpt-5")`

Asks the model for the C++ in one call and returns the code alone. `client` is an `OpenAI()` client or one of your
`frontier_clients`, and `messages` are the messages to send, such as `messages_for(...)`. Make one Chat Completions
call with the model `model` and those messages, passing everything by name, as the SDK requires. As the lesson does,
set the reasoning effort to `"high"` when `"gpt"` is in the model's name, and send none (`None`) otherwise.

Then return the code from the reply's text. Models wrap code in markdown fences, so drop every fence line, which is a
line that starts with three backticks, whatever follows them (`cpp`, `c++` or nothing), and remove the whitespace at
both ends. A reply with no fences comes back with only its whitespace removed. The lesson's two `replace` calls are
not enough: a reply fenced as `c++` would keep a stray `c++` line, which the compiler rejects.

<details><summary>Hint</summary>

```python
reasoning_effort = "high" if "gpt" in model else None
response = client.chat.completions.create(model=model, messages=messages, reasoning_effort=reasoning_effort)
reply = response.choices[0].message.content
```

Then keep the lines of `reply.splitlines()` that do not start with three backticks, join them with new lines, and
strip the result.

</details>

In [ ]:
def port(client, messages, model="gpt-5"):
    # Write port here, as described above.
    raise NotImplementedError("port is not written yet")

In [ ]:
check("port")

### 4 · `frontier_clients(env)`

Makes one `OpenAI` client for each frontier provider whose key is set, and returns them in a dict keyed by provider.
`env` is a mapping of environment variables, such as `os.environ` after `load_dotenv(override=True)`: read the keys
from `env` only. Make:

- `"openai"`: a client with the key `env["OPENAI_API_KEY"]`, at OpenAI's default endpoint;
- `"anthropic"`, `"gemini"` and `"grok"`: a client at that provider's base URL, with its key from `env`
  (`ANTHROPIC_API_KEY`, `GOOGLE_API_KEY` and `GROK_API_KEY`). The base URLs, from the lesson, are
  `https://api.anthropic.com/v1/`, `https://generativelanguage.googleapis.com/v1beta/openai/` and
  `https://api.x.ai/v1`.

Leave out any provider whose key is missing or empty. The lesson makes every client even without its key, but a
client made with no key quietly falls back to your `OPENAI_API_KEY`: it would send your OpenAI key to another
provider. Making a client sends nothing, so the checks use the real `OpenAI` here and read its base URL and key.

<details><summary>Hint</summary>

```python
OpenAI(api_key=env["OPENAI_API_KEY"])
OpenAI(base_url="https://api.x.ai/v1", api_key=env["GROK_API_KEY"])
```

`env.get(name)` is `None` for a missing key and `""` for an empty one, and both are false in an `if`.

</details>

In [ ]:
def frontier_clients(env):
    # Write frontier_clients here, as described above.
    raise NotImplementedError("frontier_clients is not written yet")

In [ ]:
check("frontier_clients")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 4, Day 3 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

A port with `gpt-5-nano`, `claude-haiku-4-5`, `gemini-3.1-flash-lite` or `grok-4-fast-non-reasoning` costs about a
cent or less; the lesson's `gpt-5` with high reasoning costs more. Compiling needs a C++ compiler. The lesson's
`clang++` flags are for a Mac; in a Linux codespace, use `g++`. Without a compiler, paste the C++ into the online
compiler the lesson suggests.

1. Load your keys with `load_dotenv(override=True)`, build `clients = frontier_clients(os.environ)`, and print which
   providers you have.
2. Get your machine's details with the lesson's helper: put the course's `week4` folder on the path with
   `sys.path.append(str(ROOT / "week4"))`, then `from system_info import retrieve_system_info`, and set
   `system_info = retrieve_system_info()`. For a Linux codespace, set
   `compile_command = ["g++", "-std=c++17", "-O3", "-march=native", "main.cpp", "-o", "main"]` and
   `run_command = ["./main"]`.
3. Run `RUMBLE` with `exec(RUMBLE, {"__builtins__": __builtins__})`, as the lesson's `run_python` does, and note its
   result and time.
4. For each provider, port `RUMBLE` with a cheap model and write the code to `main.cpp` in a new folder from
   `tempfile.mkdtemp()`. Compile and run it there, with `cwd=` that folder in each `subprocess.run` call, and compare
   its result and time with Python's.

<details><summary>Stuck? One way to write it</summary>

```python
import os
import subprocess
import sys
import tempfile

from dotenv import load_dotenv

load_dotenv(override=True)
clients = frontier_clients(os.environ)
print("Providers:", ", ".join(clients))

sys.path.append(str(ROOT / "week4"))  # the course's week4 folder, for its system_info helper
from system_info import retrieve_system_info

system_info = retrieve_system_info()
compile_command = ["g++", "-std=c++17", "-O3", "-march=native", "main.cpp", "-o", "main"]  # a Linux codespace
run_command = ["./main"]
cheap = {"openai": "gpt-5-nano", "anthropic": "claude-haiku-4-5",
         "gemini": "gemini-3.1-flash-lite", "grok": "grok-4-fast-non-reasoning"}

exec(RUMBLE, {"__builtins__": __builtins__})  # the Python version prints its result and time

messages = messages_for(RUMBLE, system_info, compile_command)
for name, client in clients.items():
    cpp = port(client, messages, model=cheap[name])
    folder = tempfile.mkdtemp()
    with open(os.path.join(folder, "main.cpp"), "w", encoding="utf-8") as f:
        f.write(cpp)
    try:
        subprocess.run(compile_command, cwd=folder, check=True, text=True, capture_output=True)
        result = subprocess.run(run_command, cwd=folder, check=True, text=True, capture_output=True)
        print(cheap[name], result.stdout, sep="\n")
    except subprocess.CalledProcessError as e:
        print(cheap[name], "failed:", e.stderr[:500], sep="\n")
    except FileNotFoundError:  # no g++ here: paste the C++ into the online compiler instead
        print(cheap[name], "has no compiler to run it; its C++:", cpp, sep="\n")
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.